# 04 · Positional Encoding

**Goal:** see why attention needs position information and how the sinusoidal encoding from the paper provides it.

Concepts: *permutation invariance*, *sinusoidal positional encoding*, *wavelengths*, *relative position*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Token** – one piece of a sentence, usually a word. Like one Lego brick in a row of bricks that spells a sentence.
- **Embedding** – a list of numbers describing a token, like a player card with stats (speed, strength, ...).
- **Position** – the slot number of a token in the sentence, like a seat number in a classroom row (0, 1, 2, ...).
- **Positional encoding** – a special list of numbers for each seat number, added to the token's card so the model knows where the token sits. Think of GPS coordinates for a word's place in the sentence.
- **Sine / cosine wave** – a smooth up-and-down wiggle between -1 and 1, like the height of a swing over time.
- **Wavelength** – how long one full wiggle takes. Short wavelength = fast wiggle (a second hand), long wavelength = slow wiggle (an hour hand).
- **Dot product** – multiply two lists of numbers entry by entry, then add up: a score for how similar the lists are.
- **Shape** – how an array is laid out, like (rows, columns) of a spreadsheet.

## 1. The problem: attention has no sense of order
Self-attention (notebook 05) treats its input as a **set**: shuffle the tokens and each token's output is just shuffled the same way. "dog bites man" and "man bites dog" would look the same.

Tiny demo: any order-blind pooling of embeddings gives identical results for shuffled sentences.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
d_model = 8
words = {w: rng.normal(size=d_model) for w in ["dog", "bites", "man"]}

s1 = ["dog", "bites", "man"]
s2 = ["man", "bites", "dog"]
sum1 = sum(words[w] for w in s1)
sum2 = sum(words[w] for w in s2)
print("identical representations:", np.allclose(sum1, sum2))

> 🧒 **In plain words:** We give each of three words ("dog", "bites", "man") a random list of 8 numbers (its "card"). `rng.normal(size=d_model)` just draws 8 random numbers; `d_model` is how many numbers each card has.
> Then we add up the cards for "dog bites man" and for "man bites dog". Adding does not care about order, just like 2 + 3 + 4 = 4 + 3 + 2.
> **Notice:** it prints `True`, meaning the two sentences got exactly the same result, even though they mean different things. That is the problem we need to fix.
> **Try it:** change `s2` to `["dog", "bites", "dog"]` and run again. You should get `False`, because now the words are different, not just reordered.

## 2. The fix: add a position-dependent vector
Give every position `pos = 0, 1, 2, …` its own vector `PE[pos]` of size `d_model` and **add** it to the token embedding.

The paper's choice (no learning needed):

$$PE_{(pos,2i)} = \sin\!\big(pos / 10000^{2i/d}\big) \qquad PE_{(pos,2i+1)} = \cos\!\big(pos / 10000^{2i/d}\big)$$

Each pair of dimensions is a sine/cosine wave with its own **wavelength**: fast-varying for small `i`, slow for large `i` — like digits of a counter (units flip fast, thousands slowly).

> 🧒 **In plain words:** Think of a car odometer or a clock. The second hand spins fast, the minute hand slower, the hour hand slowest. Reading all the hands together tells you the exact time.
> The formula does the same: every position number gets a row of sine and cosine "hands", each turning at a different speed. `pos` is the seat number, `i` picks which hand, and the big `10000` just spreads the speeds from very fast to very slow. `sin` and `cos` are the up-and-down wiggle values (between -1 and 1) of a hand's angle.
> Because the combination of all hand positions is different for every seat number, each position gets its own unique code.

In [ ]:
def positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, None]                  # (max_len, 1)
    i = np.arange(0, d_model, 2)[None, :]              # (1, d_model/2)
    angle = pos / np.power(10000.0, i / d_model)       # (max_len, d_model/2)
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe

pe = positional_encoding(6, 8)
print(pe.round(2))

> 🧒 **In plain words:** This function builds a table: one row per position, one column per number in the code.
> - `np.arange(max_len)[:, None]` makes the list 0, 1, 2, ... and stands it up as a single column. `[None, :]` does the opposite for the `i` values: it lays them as a single row.
> - Dividing a column by a row makes a whole grid at once (every position with every speed). That grid is `angle`.
> - `pe[:, 0::2]` means "all rows, every second column starting at 0" (columns 0, 2, 4, ...), which get `sin`. `pe[:, 1::2]` is the columns in between (1, 3, 5, ...), which get `cos`.
> **Notice:** row 0 is `0, 1, 0, 1, ...` (sin 0 = 0, cos 0 = 1). In the first columns the numbers change a lot from row to row; in the last columns they barely move (0.00, 1.00).
> **Try it:** change `positional_encoding(6, 8)` to `(3, 4)` and see a smaller table.

In [ ]:
pe = positional_encoding(50, 64)
plt.figure(figsize=(8, 4))
plt.imshow(pe, aspect="auto", cmap="RdBu")
plt.xlabel("dimension"); plt.ylabel("position"); plt.colorbar(); plt.title("sinusoidal positional encoding")
plt.show()

> 🧒 **In plain words:** Same table, but bigger (50 positions, 64 numbers) and drawn as a picture. `imshow` paints each number as a colour: red for one end of the scale, blue for the other, white near 0. The bar on the right is the colour key.
> **Notice:** the left side is stripy (fast wiggles), and the stripes get wider and wider towards the right (slow wiggles). It is a bit like the odometer: the right-most digits barely change.

Left columns oscillate quickly (fine position), right columns slowly (coarse position).

> 🧒 **Think of it like...** reading a clock: the second hand tells you the fine detail, the hour hand tells you roughly where in the day you are. Together they pin down the exact moment.

In [ ]:
plt.figure(figsize=(8, 3))
for dim in [0, 8, 16, 32]:
    plt.plot(pe[:, dim], label=f"dim {dim}")
plt.xlabel("position"); plt.legend(); plt.title("individual dimensions are waves with different wavelengths")
plt.show()

> 🧒 **In plain words:** We pick four columns of the table (0, 8, 16, 32) and draw each as a line going across positions. The loop `for dim in [...]` simply draws one line per column.
> **Notice:** dimension 0 wiggles quickly, and the higher the dimension number the longer and lazier the wave. That is the "different hand speeds" idea made visible.
> **Try it:** add `48` to the list `[0, 8, 16, 32]`. You should see an even flatter line.

## 3. Every position gets a unique code
No two rows are equal, so the model can tell positions apart.

In [ ]:
pe = positional_encoding(200, 64)
dists = np.linalg.norm(pe[:, None] - pe[None, :], axis=-1)
np.fill_diagonal(dists, np.inf)
print("smallest distance between any two different positions:", dists.min().round(3))

> 🧒 **In plain words:** We want to check no two positions share the same code. We measure the distance between every pair of rows, like measuring how far apart every pair of houses is on a map.
> - `pe[:, None] - pe[None, :]` subtracts every row from every other row, all at once (200 by 200 pairs).
> - `np.linalg.norm(..., axis=-1)` turns each difference into one distance number (straight-line length).
> - `np.fill_diagonal(dists, np.inf)` ignores each row compared with itself (distance 0) by setting those to infinity.
> **Notice:** the smallest distance among different positions is about 1.47. It is not zero, so every position has its own code.

## 4. Closeness encodes distance
The dot product `PE[p] · PE[q]` depends only on `|p − q|` and decays with distance, giving attention a sense of "nearby".

In [ ]:
pe = positional_encoding(100, 128)
sim = pe @ pe.T
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1); plt.imshow(sim, cmap="viridis"); plt.title("PE[p]·PE[q]"); plt.xlabel("q"); plt.ylabel("p")
plt.subplot(1, 2, 2)
for p in [10, 40, 70]:
    plt.plot(np.arange(100) - p, sim[p], label=f"p={p}")
plt.xlabel("q − p"); plt.legend(); plt.title("same shape for every p")
plt.tight_layout(); plt.show()

> 🧒 **In plain words:** `pe @ pe.T` takes the dot product (a similarity score) of every position's code with every other position's code. `.T` flips the table so rows become columns. The result `sim` is a 100 by 100 table of scores.
> The left picture shows that table. The right picture takes three rows (positions 10, 40, 70) and plots them against the gap `q - p`, how far away the other position is.
> **Notice:** the brightest spot is on the diagonal (a position is most similar to itself), and scores fade as positions get further apart. The three curves look like the same bump, just sliding along: only the distance matters, not the exact seat number. It is like friends sitting close in class being easier to talk to.

## 5. Relative shifts are linear
For any fixed offset `k`, `PE[pos+k]` is a **linear function** (a rotation) of `PE[pos]`. This makes it easy for the model to learn "attend to the token 2 steps back". Check on one frequency pair:

In [ ]:
d, k, pos, i = 64, 3, 10, 2
w = 1 / 10000 ** (2 * i / d)
pe_pos = np.array([np.sin(w * pos),       np.cos(w * pos)])
pe_pk  = np.array([np.sin(w * (pos + k)), np.cos(w * (pos + k))])
R = np.array([[ np.cos(w*k), np.sin(w*k)],
              [-np.sin(w*k), np.cos(w*k)]])        # rotation depending only on k
print("PE[pos+k]        :", pe_pk.round(4))
print("R(k) @ PE[pos]   :", (R @ pe_pos).round(4))

> 🧒 **In plain words:** We test a claim: if you know the code at position `pos`, you can get the code at `pos + k` by just turning it by a fixed angle, like rotating a clock hand. Here `k = 3` steps ahead.
> - `pe_pos` and `pe_pk` are the sin/cos pair at the two positions, worked out directly.
> - `R` is a rotation recipe (a 2 by 2 grid of numbers) that depends only on `k`, not on `pos`.
> - `R @ pe_pos` applies the rotation to `pe_pos` (`@` means matrix multiplication: mix the two numbers using the grid).
> **Notice:** both lines print the same two numbers (0.8559 and 0.5172 in the earlier run). So "3 steps ahead" is always the same turn, which is easy for a model to learn.
> **Try it:** change `k` to 5 and run: the two lines should still match.

## 6. Putting it together: embedding + position
`x = embedding(ids) * √d_model + PE[:T]` — shape `(B, T, d_model)`, with position broadcast across the batch.

In [ ]:
B, T, V, d_model = 2, 5, 13, 16
rng = np.random.default_rng(0)
E = rng.normal(0, 1/np.sqrt(d_model), size=(V, d_model))
ids = np.array([[3, 1, 4, 1, 5],
                [9, 2, 6, 5, 3]])

x = E[ids] * np.sqrt(d_model) + positional_encoding(T, d_model)   # (B,T,d)+(T,d) broadcasts
print("x shape:", x.shape)
print("token 1 appears at t=1 and t=3 in example 0:")
print("  identical now?", np.allclose(x[0, 1], x[0, 3]), "(position makes them differ)")

> 🧒 **In plain words:** Now we build the real input. `B` is the number of examples (2 sentences), `T` is the number of positions (5 tokens per sentence), `V` is the vocabulary size (13 possible tokens) and `d_model` is the card size (16).
> - `E` is a table of 13 cards, one per token.
> - `ids` holds token numbers: 2 rows (examples) by 5 columns (positions).
> - `E[ids]` swaps each token number for its card, giving shape (2, 5, 16): 2 examples, 5 positions, 16 numbers each.
> - `* np.sqrt(d_model)` scales the cards up a bit so they are not drowned out by the position codes. Then we add the position table; the (5, 16) table is reused for both examples automatically (that is "broadcasting").
> **Notice:** the shape is (2, 5, 16). Token 1 appears at positions 1 and 3 of the first sentence, and now its vectors differ (`False`), because the position code was added.

Now the "dog bites man" problem is solved: the vector the model sees for **"dog"** depends on where it sits. (Pooling by plain *sum* would still be order-blind since the PE terms add up to the same total — but attention looks at tokens individually and in pairs, so it can use position.)

In [ ]:
pe3 = positional_encoding(3, d_model)
rng = np.random.default_rng(0)
words = {w: rng.normal(size=d_model) for w in ["dog", "bites", "man"]}
dog_first = words["dog"] + pe3[0]     # "dog bites man"
dog_last  = words["dog"] + pe3[2]     # "man bites dog"
print("'dog' vector identical in both sentences?", np.allclose(dog_first, dog_last))

> 🧒 **In plain words:** Back to "dog bites man" versus "man bites dog". In the first, "dog" is at seat 0; in the second, seat 2. We add the matching seat code to the same "dog" card each time.
> **Notice:** it prints `False`: the model now sees different vectors for "dog" depending on where it sits. Problem solved.
> **Try it:** change `pe3[2]` to `pe3[0]`. Both are now at seat 0, so you should get `True`.

## Notes
- **Learned** positional embeddings (a trainable `(max_len, d)` table) work about as well; the paper tested both.
- Modern models often use *rotary* (RoPE) or *relative* schemes, but the purpose is the same.

## Recap
Attention is order-blind → add `PE[pos]` to each token vector → each position has a unique, smoothly varying code.

➡ Next: **05 · Attention**.